In [3]:
!pip install --upgrade transformers unsloth

In [7]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

# 1. Load a dedicated text-only tokenizer for the Gemma 4 architecture
# This ensures we have a .pad() method and standard text behavior
text_tokenizer = AutoTokenizer.from_pretrained("unsloth/gemma-4-E2B-it")

# 2. Ensure the tokenizer has a padding token (Standard LoRA practice)
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

# 3. Re-initialize the Trainer with this dedicated text tokenizer
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    # We pass the text_tokenizer explicitly here
    tokenizer = text_tokenizer,
    data_collator = DataCollatorForLanguageModeling(text_tokenizer, mlm=False),
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        remove_unused_columns = False,
    ),
)

# 4. Start Training!
trainer.train()

# 5. Save the result
model.save_pretrained("gemma4_lora_model")
text_tokenizer.save_pretrained("gemma4_lora_model")
print("Model saved successfully!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.
[datasets.arrow_dataset|WARNING]num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 106}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,110,080 of 5,112,407,584 (0.16% trained)


Step,Training Loss
1,10.134863
2,10.110005
3,10.119626
4,9.967802
5,9.476060
6,8.613267
7,7.529104
8,6.551352
9,5.736755
10,5.074330


Model saved successfully!
